[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nikbaya/smartbiomed_practicals_2026/blob/main/session4/practical.ipynb)

# Session 4: Fine-mapping

**Timing**: ~60 minutes (Parts 1–3 ~35 min; challenges for fast finishers).

**Dataset**: two simulated GWAS **loci** (~400 variants each, N=5,000) with realistic LD.
- Locus A has **one** causal variant hidden in a cluster of near-perfectly-correlated SNPs.
- Locus B has **two** causal variants.

The goal of fine-mapping: go from "dozens of significant variants in a peak" to a short list of
**credible** causal candidates. Everything is built from regression + a one-line Bayes factor.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SETUP — run once. Downloads the data (on Colab) and defines run_gwas + helpers.
# ─────────────────────────────────────────────────────────────────────────────
import os, numpy as np, pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 80, 'font.size': 11})

# Locate the data directory, downloading from GitHub if needed (e.g. on Colab).
import os, urllib.request
_NEED = ['finemap_data.npz']
_LFS  = {'gwas_data.npz', 'sumstats_real.npz', 'pca_data.npz', 'finemap_data.npz'}   # Git LFS
def _has_all(d):
    return d and all(os.path.exists(os.path.join(d, f)) for f in _NEED)
DATA_DIR = next((d for d in ('../data', 'data') if _has_all(d)), None)
if DATA_DIR is None:
    DATA_DIR = 'data'; os.makedirs(DATA_DIR, exist_ok=True)
    for _f in _NEED:
        _dest = os.path.join(DATA_DIR, _f)
        if os.path.exists(_dest):
            continue
        _base = ('https://media.githubusercontent.com/media' if _f in _LFS
                 else 'https://raw.githubusercontent.com')
        _url = f'{_base}/nikbaya/smartbiomed_practicals_2026/main/data/{_f}'
        print(f'Downloading {_f} from GitHub ...')
        urllib.request.urlretrieve(_url, _dest)

d = np.load(os.path.join(DATA_DIR, 'finemap_data.npz'), allow_pickle=True)
GA = d['GA'].astype(np.float32); yA = d['yA']; posA = d['posA']; causalA = int(d['causalA'][0])
GB = d['GB'].astype(np.float32); yB = d['yB']; posB = d['posB']; causalB = d['causalB']
print(f"Locus A: {GA.shape[1]} variants, N={GA.shape[0]:,}, 1 causal")
print(f"Locus B: {GB.shape[1]} variants, N={GB.shape[0]:,}, 2 causal")

def run_gwas(y, G, covars=None, chunk=5_000):
    """
    Vectorised OLS GWAS: regress phenotype y on each column of G.
    Processes variants in batches to keep peak memory usage low.

    Parameters
    ----------
    y      : (N,)    phenotype (will be mean-centred internally)
    G      : (N, M)  genotype matrix (0/1/2), NaN = missing (treated as mean)
    covars : (N, k)  covariate matrix (optional); age and sex recommended
    chunk  : int     variants per batch (default 5,000)

    Returns
    -------
    betas  : (M,)  per-variant OLS effect size estimate
    ses    : (M,)  standard error of beta
    pvals  : (M,)  two-sided p-value
    """
    N, M = G.shape
    if covars is None:
        C = np.ones((N, 1))
    else:
        C = np.column_stack([np.ones(N), covars])

    # Residualise y on covariates once (cheap)
    Q, _  = np.linalg.qr(C, mode='reduced')
    y_r   = y - Q @ (Q.T @ y)
    ss_y  = float(np.dot(y_r, y_r))
    n_df  = N - C.shape[1] - 1

    betas = np.empty(M); ses = np.empty(M)

    for s in range(0, M, chunk):
        e    = min(s + chunk, M)
        G_c  = G[:, s:e].astype(float)
        mu   = np.nanmean(G_c, axis=0)
        ri, ci = np.where(np.isnan(G_c)); G_c[ri, ci] = mu[ci]   # mean-impute
        G_r  = G_c - Q @ (Q.T @ G_c)                              # residualise
        ss_G = (G_r**2).sum(0)
        b    = G_r.T @ y_r / ss_G
        betas[s:e] = b
        rss  = ss_y - b**2 * ss_G
        ses[s:e]   = np.sqrt(np.clip(rss, 0, None) / n_df / ss_G)

    t_stats = betas / (ses + 1e-300)
    pvals   = 2 * stats.t.sf(np.abs(t_stats), df=n_df)
    return betas, ses, pvals

def r2_with(G, j):
    """r² between variant j and every variant in the locus."""
    g = G[:, j]
    return np.array([np.corrcoef(g, G[:, k])[0, 1]**2 for k in range(G.shape[1])])

print("Setup ready: run_gwas(), r2_with().")


## Part 1: Why a single GWAS peak isn't enough

Inside an associated locus, **many** variants are genome-wide significant — not because they are
all causal, but because they are in **LD** with the causal one. A LocusZoom-style plot (coloured
by $r^2$ with the lead SNP) shows the tell-tale triangular peak. The marginal **lead** SNP is often
just a *tag*, not the causal variant.


In [ ]:
# Exercise 1.1: regional GWAS + LocusZoom of Locus A
# YOUR CODE HERE — run the GWAS for the locus
betas, ses, pvals = run_gwas(???, ???)
lead = int(np.argmin(pvals))
r2   = r2_with(GA, lead)

print(f"Genome-wide significant variants: {(pvals < 5e-8).sum()} of {len(pvals)}")
print(f"Marginal lead = variant {lead} (pos {posA[lead]} kb);  true causal = variant {causalA}")

fig, ax = plt.subplots(figsize=(11, 4))
sc = ax.scatter(posA, -np.log10(pvals + 1e-300), c=r2, cmap='RdYlBu_r', vmin=0, vmax=1, s=18)
ax.scatter(posA[causalA], -np.log10(pvals[causalA] + 1e-300), marker='*', s=320,
           color='lime', edgecolor='black', zorder=5, label='true causal')
plt.colorbar(sc, ax=ax, label='$r^2$ with lead'); ax.axhline(-np.log10(5e-8), color='red', ls='--')
ax.set_xlabel('Position (kb)'); ax.set_ylabel(r'$-\log_{10}p$')
ax.set_title('Locus A — many significant variants in one LD peak'); ax.legend()
plt.tight_layout(); plt.show()
print("Q: How many variants would you have to follow up if you took every significant one?")


## Part 2: "Poor man's" fine-mapping — conditional analysis

A simple way to count **independent** signals: take the lead variant, add it as a **covariate**,
and re-run the GWAS. If a signal remains, there's a second independent association; repeat until
nothing is significant. (This isn't a credible set, but it builds the right intuition.)


In [ ]:
# Exercise 2.1: iterative conditional analysis on Locus A
signals = []
_, _, pv = run_gwas(yA, GA)
while pv.min() < 5e-8:
    lead = int(np.argmin(pv))
    signals.append(lead)
    # YOUR CODE HERE: re-run the GWAS conditioning on ALL signals found so far
    # (pass the signal genotypes as covariates)
    _, _, pv = run_gwas(yA, GA, ???)

print(f"Independent signals found: {len(signals)}  → variants {signals}")
print(f"(Locus A has 1 causal variant — do you recover a single independent signal?)")
print("Q: After conditioning on the lead, why do all the other peak variants drop out?")


## Part 3: Posterior probabilities and the credible set

Bayesian fine-mapping assigns each variant a **posterior inclusion probability (PIP)** — the
probability it is the causal one. Assuming a single causal variant in the locus, Wakefield's
**approximate Bayes factor (ABF)** for variant $j$ uses only its $z = \hat\beta/\text{se}$:

$$\log\text{ABF}_j = \tfrac{1}{2}\log(1-r) + \tfrac{1}{2} z_j^2\, r, \qquad r = \frac{W}{V_j + W}$$

where $V_j=\text{se}_j^2$ and $W$ is the prior variance of the effect. Normalising the ABFs gives
PIPs; the **95% credible set** is the smallest set of variants whose PIPs sum to ≥ 0.95.


In [ ]:
# Exercise 3.1: PIP and the 95% credible set for Locus A
betas, ses, pvals = run_gwas(yA, GA)
z = betas / ses
W = 0.04                          # prior variance of the effect size
V = ses**2
r = W / (V + W)

# YOUR CODE HERE
log_abf = ???                     # 0.5*log(1-r) + 0.5*z^2*r
abf     = np.exp(log_abf - log_abf.max())   # stabilised
pip     = ???                     # normalise to sum to 1

# 95% credible set: smallest set of top-PIP variants with cumulative PIP >= 0.95
order = np.argsort(pip)[::-1]
cum   = np.cumsum(pip[order])
cred  = order[:np.searchsorted(cum, 0.95) + 1]

print(f"95% credible set: {sorted(cred.tolist())}  ({len(cred)} variants)")
print(f"Contains the true causal ({causalA})? {causalA in cred.tolist()}")
print(f"PIP of the true causal: {pip[causalA]:.2f}   (max PIP in locus: {pip.max():.2f})")
print("Q: Fine-mapping cut the significant peak down to how few variants?")


---

## Challenge Questions


### Challenge 1: A PIP LocusZoom — ~10 min

Re-draw the locus with each variant's height = PIP (instead of −log10 p). The credible set should
stand out as the handful of high-PIP variants. Mark the true causal.


In [ ]:
# Challenge: PIP LocusZoom for Locus A  (reuse pip, cred from Part 3)
fig, ax = plt.subplots(figsize=(11, 4))
in_cs = np.zeros(len(pip), bool); in_cs[cred] = True
ax.scatter(posA[~in_cs], pip[~in_cs], s=15, color='lightgrey', label='not in credible set')
ax.scatter(posA[in_cs], pip[in_cs], s=40, color='#e15759', zorder=4, label='95% credible set')
ax.scatter(posA[causalA], pip[causalA], marker='*', s=320, color='lime',
           edgecolor='black', zorder=5, label='true causal')
ax.set_xlabel('Position (kb)'); ax.set_ylabel('PIP'); ax.set_title('Fine-mapping: PIP by position')
ax.legend(); plt.tight_layout(); plt.show()
print(f"Sum of PIP over the credible set: {pip[cred].sum():.2f}")
print("Q: The whole significant peak collapses to a few high-PIP variants — why those?")


### Challenge 2: Resolution depends on power — ~10 min

Fine-mapping resolution improves with sample size. Recompute the credible set on random subsets of
the individuals (N/2, N/4) and watch it **grow** as power falls.


In [ ]:
# Challenge: credible-set size vs sample size
def credible_set(y, G, W=0.04):
    b, se, _ = run_gwas(y, G)
    z = b / se; V = se**2; rr = W / (V + W)
    la = 0.5*np.log(1-rr) + 0.5*z**2*rr
    p = np.exp(la - la.max()); p /= p.sum()
    o = np.argsort(p)[::-1]; c = np.cumsum(p[o])
    return o[:np.searchsorted(c, 0.95) + 1]

rng = np.random.default_rng(0)
for frac in [1.0, 0.5, 0.25]:
    n = int(frac * len(yA)); idx = rng.choice(len(yA), n, replace=False)
    # YOUR CODE HERE: credible set on this subsample
    cs = credible_set(???, ???)
    print(f"N={n:5d}: credible-set size = {len(cs):3d}, contains causal = {causalA in cs.tolist()}")
print("Q: Why does a smaller sample give a larger (less useful) credible set?")


### Challenge 3: When one causal variant isn't enough — ~12 min

The single-causal credible set assumes exactly **one** causal variant per locus. **Locus B** has
two. Build the single-causal credible set for Locus B and check whether it captures **both**
causals — then use conditional analysis to recover the second signal (the idea behind SuSiE).


In [ ]:
# Challenge: two causal variants (Locus B)
b, se, pv = run_gwas(yB, GB)
z = b/se; W=0.04; V=se**2; rr=W/(V+W)
la = 0.5*np.log(1-rr)+0.5*z**2*rr; pip_b = np.exp(la-la.max()); pip_b/=pip_b.sum()
o=np.argsort(pip_b)[::-1]; cum=np.cumsum(pip_b[o]); cred_b=o[:np.searchsorted(cum,0.95)+1]
print(f"True causals: {causalB.tolist()}")
print(f"Single-causal 95% credible set: {sorted(cred_b.tolist())}")
print(f"  captures both causals? {all(c in cred_b.tolist() for c in causalB)}")

# YOUR CODE HERE: condition on the lead, then refit — does the SECOND causal now stand out?
lead = int(np.argmin(pv))
_, _, pv2 = run_gwas(yB, GB, ???)
print(f"After conditioning on variant {lead}: new lead = {int(np.argmin(pv2))} "
      f"(near the other causal {causalB.tolist()})")
print("Q: Why does a single-causal credible set fail when there are two signals?")
